# Notebook 3 — elementos finitos

Minicurso **Da kalimba à viga** · Bloco 4

Quatro etapas: montar, impor o engaste, resolver o problema de autovalores e estudar a
convergência. No fim, o seu código estará validado contra a solução analítica do bloco 2.

## Etapa 1 — montagem

In [ ]:
import numpy as np

def matrizes_elemento(EI, rhoA, l):
    """Matrizes de rigidez e de massa de um elemento de viga de Hermite."""
    K = EI/l**3 * np.array([
        [ 12,    6*l,   -12,    6*l  ],
        [ 6*l,   4*l*l, -6*l,   2*l*l],
        [-12,   -6*l,    12,   -6*l  ],
        [ 6*l,   2*l*l, -6*l,   4*l*l]])
    M = rhoA*l/420 * np.array([
        [156,   22*l,   54,   -13*l  ],
        [22*l,   4*l*l, 13*l,  -3*l*l],
        [54,    13*l,  156,   -22*l  ],
        [-13*l, -3*l*l,-22*l,   4*l*l]])
    return K, M


def monta(ne, L, EI, rhoA):
    """Monta K e M globais de uma viga dividida em ne elementos iguais."""
    l = L / ne
    n = 2 * (ne + 1)                      # 2 graus de liberdade por no
    K = np.zeros((n, n))
    M = np.zeros((n, n))
    for e in range(ne):
        Ke, Me = matrizes_elemento(EI, rhoA, l)
        gdl = [2*e, 2*e + 1, 2*e + 2, 2*e + 3]
        K[np.ix_(gdl, gdl)] += Ke
        M[np.ix_(gdl, gdl)] += Me
    return K, M

**Confira a montagem antes de seguir.** A matriz tem de ser simétrica, e a
massa total guardada em `M` tem de ser a massa da viga.

In [ ]:
L, h, b = 0.040, 0.001, 0.010
E, rho = 200e9, 7850
I, A = b*h**3/12, b*h
EI, rhoA = E*I, rho*A

K, M = monta(4, L, EI, rhoA)
print("simetrica:", np.allclose(K, K.T), np.allclose(M, M.T))

# a massa total aparece somando os termos de translacao
desloc = np.arange(0, M.shape[0], 2)
print("massa montada:", round(M[np.ix_(desloc, desloc)].sum(), 8), "kg")
print("massa real   :", round(rho*A*L, 8), "kg")

## Etapa 2 — contorno e solução

O engaste em $x=0$ significa $w_1 = \theta_1 = 0$: basta eliminar as duas primeiras linhas
e colunas. Sem isso aparecem dois autovalores nulos, de translação e rotação de corpo
rígido.

In [ ]:
from scipy.linalg import eigh

ne = 10
K, M = monta(ne, L, EI, rhoA)
Kc, Mc = K[2:, 2:], M[2:, 2:]      # engaste

lam, vec = eigh(Kc, Mc)            # resolve K phi = lambda M phi
f = np.sqrt(lam) / (2*np.pi)       # frequencias em Hz
print(np.round(f[:4], 2))

## Etapa 3 — verificação contra o analítico

In [ ]:
BL = np.array([1.8751041, 4.6940911, 7.8547574])
exato = BL**2 / (2*np.pi*L**2) * np.sqrt(EI/rhoA)
print("analitico:", np.round(exato, 2))

for ne in (2, 10, 50):
    K, M = monta(ne, L, EI, rhoA)
    lam, _ = eigh(K[2:, 2:], M[2:, 2:])
    f = np.sqrt(lam[:3]) / (2*np.pi)
    erro = 100 * (f - exato) / exato
    print(f"ne={ne:3d}  f={np.round(f,2)}  erro={np.round(erro,3)} %")

Repare que o MEF dá sempre frequências **maiores** que as exatas. Isso não é
coincidência: restringir a solução a um conjunto de funções equivale a acrescentar rigidez,
e um sistema mais rígido vibra mais depressa.

### Os modos numéricos por cima dos analíticos

In [ ]:
import matplotlib.pyplot as plt

def forma(x, bL):
    s = (np.cos(bL) + np.cosh(bL)) / (np.sin(bL) + np.sinh(bL))
    return (np.cosh(bL*x) - np.cos(bL*x)
            - s * (np.sinh(bL*x) - np.sin(bL*x)))

ne = 8
K, M = monta(ne, L, EI, rhoA)
lam, vec = eigh(K[2:, 2:], M[2:, 2:])
xn = np.linspace(0, 1, ne+1)
x = np.linspace(0, 1, 400)

plt.figure(figsize=(9, 5))
for n in range(3):
    phi = np.concatenate(([0, 0], vec[:, n]))
    w = phi[0::2]                              # so os deslocamentos
    w = w / np.max(np.abs(w))
    ya = forma(x, BL[n]); ya = ya / np.max(np.abs(ya))
    if np.dot(w[1:], np.interp(xn[1:], x, ya)) < 0:
        w = -w                                 # mesma orientacao
    plt.plot(x, ya, color=f"C{n}", label=f"modo {n+1} analitico")
    plt.plot(xn, w, "o--", color=f"C{n}", alpha=0.7, label=f"modo {n+1} MEF")

plt.axhline(0, color="0.8", lw=1)
plt.xlabel("x / L"); plt.ylabel("amplitude normalizada")
plt.legend(ncol=2, fontsize=8); plt.tight_layout(); plt.show()

## Etapa 4 — convergência

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

nes = np.arange(1, 25)
erros = []
for ne in nes:
    K, M = monta(ne, L, EI, rhoA)
    lam, _ = eigh(K[2:, 2:], M[2:, 2:])
    f = np.sqrt(lam) / (2*np.pi)
    erros.append([abs(f[i] - exato[i])/exato[i] if i < len(f) else np.nan
                  for i in range(3)])
erros = np.array(erros)

plt.figure(figsize=(8, 5))
for i in range(3):
    plt.loglog(nes, erros[:, i], "o-", label=f"modo {i+1}")
plt.axhline(0.01, color="0.7", ls="--")
plt.text(1.1, 0.012, "1 %", color="0.4")
plt.xlabel("numero de elementos"); plt.ylabel("erro relativo")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

for i in range(3):
    ok = nes[erros[:, i] < 0.01]
    print(f"modo {i+1}: converge a 1 % com {ok[0]} elementos")

## Se sobrar tempo

Três variações que a fórmula analítica não faz, e o MEF faz sem mudar de ideia.

In [ ]:
# (i) massa concentrada na ponta
def com_massa_na_ponta(ne, m_ponta):
    K, M = monta(ne, L, EI, rhoA)
    M[-2, -2] += m_ponta              # o deslocamento do ultimo no
    lam, _ = eigh(K[2:, 2:], M[2:, 2:])
    return np.sqrt(lam[:3]) / (2*np.pi)

massa_viga = rho*A*L
for frac in (0, 0.1, 0.5, 1.0):
    f = com_massa_na_ponta(20, frac*massa_viga)
    print(f"m_ponta = {frac:4.1f} m_viga -> f1 = {f[0]:8.1f} Hz")

In [ ]:
# (ii) livre-livre: nao elimina nada, e aparecem dois autovalores nulos
K, M = monta(20, L, EI, rhoA)
lam, _ = eigh(K, M)
f = np.sqrt(np.maximum(lam, 0)) / (2*np.pi)
print("tres primeiros autovalores:", np.round(f[:3], 3))
print("razoes dos modos elasticos:", np.round(f[2:6]/f[2], 3))
# esperado: 1  2.757  5.404  8.933

In [ ]:
# (iii) secao variavel: cada elemento com o seu I
def monta_variavel(ne, L, E, rho, b, alturas):
    l = L/ne
    n = 2*(ne+1)
    K = np.zeros((n, n)); M = np.zeros((n, n))
    for e in range(ne):
        h_e = alturas[e]
        Ke, Me = matrizes_elemento(E*b*h_e**3/12, rho*b*h_e, l)
        gdl = [2*e, 2*e+1, 2*e+2, 2*e+3]
        K[np.ix_(gdl, gdl)] += Ke
        M[np.ix_(gdl, gdl)] += Me
    return K, M

ne = 20
# lamina com rebaixo no meio, como numa marimba
alturas = h * (1 - 0.35*np.exp(-((np.arange(ne)+0.5)/ne - 0.5)**2 / 0.02))
K, M = monta_variavel(ne, L, E, rho, b, alturas)
lam, _ = eigh(K[2:, 2:], M[2:, 2:])
f = np.sqrt(lam[:3]) / (2*np.pi)
print("com rebaixo :", np.round(f, 1))
print("razao f2/f1 :", round(f[1]/f[0], 3), " (uniforme daria 6.267)")